In [1]:
import re
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer


In [2]:
liar_train = pd.read_csv("../data/liar/liar_train_clean.csv")
liar_valid = pd.read_csv("../data/liar/liar_valid_clean.csv")
liar_test  = pd.read_csv("../data/liar/liar_test_clean.csv")

isot = pd.read_csv("../data/isot/isot_clean.csv")


In [3]:
def clean_text(text: str) -> str:
    """
    Minimal, consistent cleaning for classical ML (TF-IDF).
    Keep it simple to avoid removing useful signal.
    """
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", "", text)     # remove URLs
    text = re.sub(r"[^a-z\s]", " ", text)           # keep letters + spaces
    text = re.sub(r"\s+", " ", text).strip()        # normalise whitespace
    return text


In [4]:
# 🔹 Step 1: Ensure no NaNs and enforce string type
for df in [liar_train, liar_valid, liar_test, isot]:
    df["statement"] = df["statement"].fillna("").astype(str)

# 🔹 Step 2: Apply cleaning function
for df in [liar_train, liar_valid, liar_test]:
    df["statement"] = df["statement"].apply(clean_text)

isot["statement"] = isot["statement"].apply(clean_text)

In [5]:
print("LIAR train:", liar_train.shape)
print("LIAR valid:", liar_valid.shape)
print("LIAR test :", liar_test.shape)
print("ISOT      :", isot.shape)

print("\nSample cleaned LIAR text:\n", liar_train["statement"].iloc[0])
print("\nSample cleaned ISOT text:\n", isot["statement"].iloc[0])

LIAR train: (10240, 2)
LIAR valid: (1284, 2)
LIAR test : (1267, 2)
ISOT      : (44898, 2)

Sample cleaned LIAR text:
 says the annies list political group supports third trimester abortions on demand

Sample cleaned ISOT text:
 st century wire says ben stein reputable professor from pepperdine university also of some hollywood fame appearing in tv shows and films such as ferris bueller s day off made some provocative statements on judge jeanine pirro s show recently while discussing the halt that was imposed on president trump s executive order on travel stein referred to the judgement by the th circuit court in washington state as a coup d tat against the executive branch and against the constitution stein went on to call the judges in seattle political puppets and the judiciary political pawns watch the interview below for the complete statements and note the stark contrast to the rhetoric of the leftist media and pundits who neglect to note that no court has ever blocked any preside

In [6]:
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2
)

In [7]:
X_liar_train = tfidf.fit_transform(liar_train["statement"])
y_liar_train = liar_train["label"].astype(int)

X_liar_valid = tfidf.transform(liar_valid["statement"])
y_liar_valid = liar_valid["label"].astype(int)

X_liar_test  = tfidf.transform(liar_test["statement"])
y_liar_test  = liar_test["label"].astype(int)

print("TF-IDF (LIAR) feature matrix shapes:")
print("Train:", X_liar_train.shape)
print("Valid:", X_liar_valid.shape)
print("Test :", X_liar_test.shape)

TF-IDF (LIAR) feature matrix shapes:
Train: (10240, 5000)
Valid: (1284, 5000)
Test : (1267, 5000)


In [8]:
from sklearn.model_selection import train_test_split

isot_train, isot_temp = train_test_split(
    isot, test_size=0.3, random_state=42, stratify=isot["label"]
)

isot_valid, isot_test = train_test_split(
    isot_temp, test_size=0.5, random_state=42, stratify=isot_temp["label"]
)

print("ISOT splits:")
print("Train:", isot_train.shape)
print("Valid:", isot_valid.shape)
print("Test :", isot_test.shape)

ISOT splits:
Train: (31428, 2)
Valid: (6735, 2)
Test : (6735, 2)


In [9]:
tfidf_isot = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2
)

X_isot_train = tfidf_isot.fit_transform(isot_train["statement"])
y_isot_train = isot_train["label"].astype(int)

X_isot_valid = tfidf_isot.transform(isot_valid["statement"])
y_isot_valid = isot_valid["label"].astype(int)

X_isot_test  = tfidf_isot.transform(isot_test["statement"])
y_isot_test  = isot_test["label"].astype(int)

print("TF-IDF (ISOT) feature matrix shapes:")
print("Train:", X_isot_train.shape)
print("Valid:", X_isot_valid.shape)
print("Test :", X_isot_test.shape)

TF-IDF (ISOT) feature matrix shapes:
Train: (31428, 5000)
Valid: (6735, 5000)
Test : (6735, 5000)


In [10]:
import os
from scipy.sparse import save_npz
import joblib

os.makedirs("results/features", exist_ok=True)

# Save LIAR features + vectoriser
save_npz("results/features/X_liar_train.npz", X_liar_train)
save_npz("results/features/X_liar_valid.npz", X_liar_valid)
save_npz("results/features/X_liar_test.npz",  X_liar_test)
joblib.dump(y_liar_train, "results/features/y_liar_train.pkl")
joblib.dump(y_liar_valid, "results/features/y_liar_valid.pkl")
joblib.dump(y_liar_test,  "results/features/y_liar_test.pkl")
joblib.dump(tfidf, "results/features/tfidf_liar.joblib")

# Save ISOT splits (raw text) as CSV for later reproducibility
isot_train.to_csv("results/features/isot_train.csv", index=False)
isot_valid.to_csv("results/features/isot_valid.csv", index=False)
isot_test.to_csv("results/features/isot_test.csv", index=False)

# Save ISOT features + vectoriser
save_npz("results/features/X_isot_train.npz", X_isot_train)
save_npz("results/features/X_isot_valid.npz", X_isot_valid)
save_npz("results/features/X_isot_test.npz",  X_isot_test)
joblib.dump(y_isot_train, "results/features/y_isot_train.pkl")
joblib.dump(y_isot_valid, "results/features/y_isot_valid.pkl")
joblib.dump(y_isot_test,  "results/features/y_isot_test.pkl")
joblib.dump(tfidf_isot, "results/features/tfidf_isot.joblib")

print("Saved TF-IDF features + labels + vectorisers into results/features/")

Saved TF-IDF features + labels + vectorisers into results/features/


In [11]:
# LIAR -> ISOT (use LIAR vectoriser)
X_isot_test_as_liar = tfidf.transform(isot_test["statement"])

# ISOT -> LIAR (use ISOT vectoriser)
X_liar_test_as_isot = tfidf_isot.transform(liar_test["statement"])

print("Cross-domain transformed shapes:")
print("ISOT test as LIAR features:", X_isot_test_as_liar.shape)
print("LIAR test as ISOT features:", X_liar_test_as_isot.shape)

Cross-domain transformed shapes:
ISOT test as LIAR features: (6735, 5000)
LIAR test as ISOT features: (1267, 5000)
